# 03 — Allocation diagnostic

Run all five diagnostic sections (combined-book exposure, correlation,
redundancy, risk contribution, benchmark comparison) on the resolved book
and export `reports/allocation_diagnostic.html`.


In [ ]:
from datetime import date
from pathlib import Path
import warnings

import pandas as pd

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.diagnostic import (
    benchmark_comparison, combined_exposure, combined_exposure_figure,
    correlation_figure, correlation_matrix, redundancy_pairs,
    render_html_report, risk_contribution,
)
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.data.providers import YahooFinanceProvider

from hailmary.allocation.returns import last_business_day_on_or_before

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/allocation_diagnostic.html')
START = date(2022, 1, 1)
END = last_business_day_on_or_before(date.today())
REDUNDANCY_THRESHOLD = 0.85

# ALIGN_WINDOW = True  (default, recommended)
#   Combined-book metrics + windowed table use the common-history window
#   (latest first-data date across all HOLDING portfolios). Apples-to-apples,
#   shorter history, static weights.
# ALIGN_WINDOW = False
#   Full available history with dynamic per-timestep weight renormalisation.
#   Longer history but early dates use only the older portfolios at boosted
#   weights, so 'early book' ≠ 'today's book'. Useful if you want pre-2024
#   context at the cost of mixing weight regimes.
ALIGN_WINDOW = True

# Annual-return target for the traffic-light styling on the benchmark table's
# 'Ann. return' column. Green ≥ target, yellow [0..target), red < 0.
# High-Sharpe-low-return rows (e.g. Simple SGD with Sharpe ~4 and ann return 1.5%)
# will be yellow under a 5% target — they're risk-adjusted-great but won't grow
# wealth at your target rate.
TARGET_ANN_RETURN = 0.05

# Date the portfolio-reconciliation section projects to. Defaults to the last
# business day. Stashaway's app values are sometimes delayed by a day —
# if the app shows 'as of 22 May' while today is 24 May, set this to
# date(2026, 5, 22) to get an exact apples-to-apples reconcile.
RECONCILE_AS_OF = END

print(f'window: {START}..{END}  |  align_window={ALIGN_WINDOW}  |  target_ann_return={TARGET_ANN_RETURN:.1%}  |  reconcile_as_of={RECONCILE_AS_OF}')

## Parse + tag + fetch returns

In [ ]:
parsed = parse_statement(STATEMENT_PATH, use_cache=False)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
holding = [p for p in portfolios if Role.HOLDING in p.roles]
tickers = sorted({
    h.metadata.ticker for p in holding for h in p.holdings
    if not h.metadata.ticker.startswith('CASH_')
})
provider = YahooFinanceProvider()
returns = provider.get_returns(tickers, START, END)
print(f'Resolved {len(portfolios)} portfolios; fetched {returns.shape[1]} ticker series')

## Fetch USDSGD (daily series for return FX adjustment)

In [ ]:
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
# Yahoo's USDSGD is labelled in UK time, so its 'closing' rate lands ~7h
# before Stashaway's Singapore-EOD snapshot. For statement-date AUM we use
# the rate parsed from the PDF (attached to portfolio.metadata via from_parsed).
# Daily series is still used for compounding return adjustments — day-over-day
# moves are roughly the same despite the timezone shift.
stashaway_fx = portfolios[0].metadata.get('statement_fx_usd_sgd')
yahoo_spot = float(fx_series_usd_sgd.iloc[-1])
print('Statement-date FX (from PDF, used for AUM):  1 USD = {:.4f} SGD'.format(stashaway_fx))
print('Yahoo USDSGD spot ({}, used for daily series): 1 USD = {:.4f} SGD'.format(
    fx_series_usd_sgd.index.max().date(), yahoo_spot,
))

## Validation — no portfolio silently dropped + window summary

In [ ]:
from hailmary.allocation.diagnostic import _build_returns_panel, PortfolioDroppedError
try:
    _validation_panel = _build_returns_panel(
        holding, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd, strict=True,
    )
    print(f'All {len(holding)} HOLDING portfolios resolved cleanly '
          f'({_validation_panel.shape[1]} series, {_validation_panel.shape[0]:,} dates).')
except PortfolioDroppedError as exc:
    print('FAIL — would drop portfolios:')
    for name, reason in exc.dropped:
        print(f'  {name}: {reason}')
    raise

first_dates = (
    _validation_panel.apply(lambda c: c.dropna().index.min().date())
    .sort_values(ascending=False)
)
common_start = first_dates.iloc[0]
aligned_days = len(_validation_panel.loc[str(common_start):].dropna(how='any'))
print()
print(f'Common-history window: [{common_start}..{END}] ({aligned_days:,} aligned dates).')
print('Per-portfolio first-data dates (latest first — these are what shrink the window):')
for name, first_date in first_dates.head(8).items():
    marker = '  ← constrains common_start' if first_date == common_start else ''
    print(f'  {name:<22} {first_date}{marker}')
if len(first_dates) > 8:
    remaining = first_dates.iloc[8:]
    print(f'  ...{len(remaining)} more portfolios start between '
          f'{remaining.min()} and {remaining.max()}')
print()
print('Set ALIGN_WINDOW=True (default) → combined-book metrics use this aligned window.')
print('Set ALIGN_WINDOW=False → full per-portfolio histories with dynamic-renorm weights.')

## Combined-book exposure

In [ ]:
exposure = combined_exposure(portfolios)
for dim, df in exposure.items():
    print(f'\n--- {dim.replace("_", " ").title()} ---')
    display(df)
combined_exposure_figure(exposure)

## Correlation matrix

In [ ]:
corr = correlation_matrix(portfolios, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd)
display(corr.round(3))
correlation_figure(corr)

## Redundancy (threshold default 0.85)

In [ ]:
pairs = redundancy_pairs(corr, threshold=REDUNDANCY_THRESHOLD, portfolios=portfolios)
if pairs:
    pd.DataFrame(pairs, columns=['a', 'b', 'rho', 'candidate'])
else:
    print(f'No portfolio pairs above ρ = {REDUNDANCY_THRESHOLD}.')
    print('If your customs are uncorrelated by design, this is expected — drop the threshold to 0.7 to surface near-redundancy.')

## Risk contribution

In [ ]:
risk = risk_contribution(portfolios, returns=returns)
print('--- By portfolio ---')
display(risk['by_portfolio'].round(4))
print('--- By holding (top 15) ---')
display(risk['by_holding'].head(15).round(4))

## Benchmark comparison

In [ ]:
bench = benchmark_comparison(portfolios, returns=returns, fx_series_usd_sgd=fx_series_usd_sgd)
bench.round(3)

## Export HTML report (strict — fails if any portfolio would drop)

In [ ]:
out = render_html_report(
    portfolios,
    REPORT_PATH,
    returns=returns,
    redundancy_threshold=REDUNDANCY_THRESHOLD,
    fx_series_usd_sgd=fx_series_usd_sgd,
    align_window=ALIGN_WINDOW,
    target_ann_return=TARGET_ANN_RETURN,
    reconciliation_as_of=RECONCILE_AS_OF,
    title='Stashaway book — allocation diagnostic',
)  # fx_rate_usd_sgd left default → picks up Stashaway PDF rate from portfolio.metadata
print(f'Wrote {out.resolve()}')